# §1.4.4 — 구조가 표본 효율을 지배한다

> 딥러닝 교재 · 1부 1장 4절 4항 (🐍)
> 선행: §1.4.1(가설 공간과 도달 가능 집합) · §1.4.2(귀납 편향의 세 출처) · §1.4.3(다항 회귀 손계산)

## 이 노트북이 답하는 질문

1. **§1.4.3의 손계산이 맞는가?** 세 점 예제를 코드로 재현해 손으로 구한 값과 대조한다.
2. **구조만 바꾸면 얼마나 달라지는가?** 자료·손실·최적화기를 완전히 고정하고 가설 공간만 바꿔 **표본 효율**을 잰다.
3. **CNN이 좋은 것인가, CNN의 편향이 이 문제에 맞는 것인가?** 화소를 고정 순열로 섞어 판정한다.

**예상 실행 시간** CPU 단일 코어 약 90초 (`FAST = True`이면 약 30초).

§1.4.2는 귀납 편향의 출처를 셋으로 나눴다. ②(목적함수)는 §1.2.4가, ③(최적화)은 §1.4.1 3절이 이미 실증했다.
**이 노트북은 ①(구조)을 맡는다.** 세 칸 중 마지막이 여기서 채워진다.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False
SEED     = 20260801
IMG      = 10      # 이미지 한 변
AMP      = 6.0     # 무늬의 세기. 작을수록 과제가 어려워진다
SAVE_PDF = False
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

CB = ['#000000','#E69F00','#56B4E9','#009E73','#D55E00','#0072B2','#CC79A7','#F0E442']
plt.rcParams.update({'figure.dpi':120,'font.size':10,'axes.grid':True,'grid.alpha':0.3,
                     'axes.prop_cycle':plt.cycler(color=CB),'figure.autolayout':True})
import matplotlib.font_manager as fm
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic','Malgun Gothic','AppleGothic','Noto Sans CJK KR',
                            'Noto Sans KR','NanumBarunGothic','Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en
_fi = [0]
def show(name):
    _fi[0] += 1
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        plt.savefig(os.path.join(FIG_DIR, f'fig_1_4_4_{_fi[0]}_{name}.pdf'),
                    bbox_inches='tight', pad_inches=0.02)
    plt.show()

print(f"numpy {np.__version__} | FAST={FAST} | 한글폰트: {KO_FONT or '없음(영문 라벨)'}")

---
## 1. §1.4.3 손계산 재현

세 점 $(-1, 0)$, $(0, 1)$, $(1, 3)$에 다항식을 적합했다. 손으로 구한 값은 이랬다.

| 차수 | 잔차제곱합 | $f(2)$ |
|---|---|---|
| 0 | $14/3$ | $4/3$ |
| 1 | $1/6$ | $13/3$ |
| 2 | $0$ | $6$ |
| 3 (표준 기저, 최소 노름) | $0$ | $21/2$ |
| 3 (체비셰프 기저, 최소 노름) | $0$ | $24$ |

`np.linalg.pinv`가 최소 노름 최소제곱해를 주므로 §1.4.3 4절의 경사하강 극한과 같다.
**손계산이 코드의 검산 기준이다.** 어긋나면 둘 중 하나가 틀린 것이다.

In [ ]:
X3 = np.array([-1.0, 0.0, 1.0])
Y3 = np.array([ 0.0, 1.0, 3.0])

def mono_basis(x, k):
    return np.stack([np.asarray(x)**j for j in range(k+1)], axis=1)

def cheb_basis(x, k):
    x = np.asarray(x, dtype=float)
    T = [np.ones_like(x), x]
    for j in range(2, k+1):
        T.append(2*x*T[-1] - T[-2])
    return np.stack(T[:k+1], axis=1)

print(" 차수      계수 (표준 기저)              RSS      f(2)")
expect = {0:(14/3, 4/3), 1:(1/6, 13/3), 2:(0.0, 6.0), 3:(0.0, 10.5)}
for k in range(4):
    A = mono_basis(X3, k)
    w = np.linalg.pinv(A) @ Y3
    rss = float(np.sum((A @ w - Y3)**2))
    f2  = float((mono_basis([2.0], k) @ w)[0])
    print(f"  {k}   {np.round(w,4)!s:<28} {rss:8.5f}  {f2:8.4f}")
    assert np.isclose(rss, expect[k][0], atol=1e-9), k
    assert np.isclose(f2,  expect[k][1], atol=1e-9), k

# 체비셰프 기저에서의 최소 노름 해 → 단항식 계수로 환산
wc = np.linalg.pinv(cheb_basis(X3, 3)) @ Y3
M  = np.array([[1,0,-1, 0],      # T0..T3 를 1, x, x^2, x^3 계수로
               [0,1, 0,-3],
               [0,0, 2, 0],
               [0,0, 0, 4]], dtype=float)
w_cheb = M @ wc
f2_cheb = float((mono_basis([2.0], 3) @ w_cheb)[0])
print(f"\n체비셰프 계수 {np.round(wc,4)}  ->  단항식 {np.round(w_cheb,4)}   f(2)={f2_cheb:.4f}")
assert np.allclose(w_cheb, [1.0, -1.5, 0.5, 3.0]) and np.isclose(f2_cheb, 24.0)
print("\n손계산과 전부 일치. §1.4.3의 다섯 값이 확인되었다.")
print("-> 같은 H, 같은 손실, 같은 자료, 같은 알고리즘인데 기저만 바꾸면 f(2)가 10.5에서 24로 바뀐다.")

---
## 2. 고전판 한 장

§1.4.3은 세 점으로 줄여 계산이 끝까지 되게 했다. 대신 잃은 것이 **시각적 인상**이므로 여기서 한 번 보충한다.
잡음 섞인 사인 곡선에 차수를 바꿔 적합한다. 새 주장은 없고, §1.4.3에서 계산한 것이 규모를 키우면 이렇게 보인다는 것뿐이다.

In [ ]:
rng = np.random.default_rng(SEED)
n_cls = 12
x_cls = np.sort(rng.uniform(0, 1, n_cls))
f_true = lambda x: np.sin(2*np.pi*np.asarray(x))
y_cls = f_true(x_cls) + rng.normal(0, 0.25, n_cls)
xg = np.linspace(-0.02, 1.02, 400)

fig, axes = plt.subplots(1, 3, figsize=(10.2, 3.1), sharey=True)
for ax, k in zip(axes, [1, 3, 9]):
    w = np.linalg.pinv(mono_basis(x_cls, k)) @ y_cls
    ax.plot(xg, f_true(xg), color=CB[0], lw=1.0, ls='--',
            label=lab('참 함수', 'true function'))
    ax.plot(xg, mono_basis(xg, k) @ w, color=CB[4], lw=1.8,
            label=lab('적합', 'fit'))
    ax.plot(x_cls, y_cls, 'o', ms=5, color=CB[5], label=lab('자료', 'data'))
    rss = np.sum((mono_basis(x_cls, k) @ w - y_cls)**2)
    ax.set_title(lab(f'차수 {k}   RSS = {rss:.3f}', f'degree {k}   RSS = {rss:.3f}'), fontsize=10)
    ax.set_xlabel('$x$'); ax.set_ylim(-2.2, 2.2)
axes[0].set_ylabel('$y$'); axes[0].legend(fontsize=7.5)
fig.suptitle(lab('훈련 오차는 차수에 대해 단조 감소한다 — 그래서 모형 선택 기준이 될 수 없다',
                 'training error decreases monotonically in degree'), y=1.03, fontsize=10)
show('classic_polynomial')

> $H_k \subseteq H_{k+1}$ 이므로 RSS가 차수에 대해 단조 감소하는 것은 자명하다.
> 따라서 RSS를 최소화하는 차수는 언제나 가장 큰 차수이며, 이는 §1.1.2의 낙관 편향이 극단적으로 나타난 형태다.

---
## 3. 과제 — 구조의 차이가 드러나는 최소 설계

$10 \times 10$ 이미지에 $3 \times 3$ 무늬 하나를 **무작위 위치**에 찍는다. 무늬는 두 종류(가로 막대 / 세로 막대)이고,
찍을 때 **부호를 무작위로** 뒤집는다. 배경은 표준정규 잡음이다. 맞힐 것은 어느 무늬였는가이다.

$$x = \varepsilon + A \cdot s \cdot \mathrm{stamp}_{r,c}(T_y), \qquad s \in \{-1, +1\} \text{ 균등}, \qquad \varepsilon \sim \mathcal{N}(0, I)$$

설계에 들어간 성질 셋이 각각 하나씩 일을 한다.

| 성질 | 무엇을 요구하는가 |
|---|---|
| 무늬가 **국소적** | 3×3 이웃만 보면 된다 |
| 위치가 **무작위** | 어디서든 같은 판정을 해야 한다 (평행이동 불변) |
| 부호가 **무작위** | 선형 상관으로는 안 되고 $\lvert \cdot \rvert$ 류의 비선형이 필요하다 |

앞의 둘은 합성곱의 귀납 편향(국소성 · 가중치 공유 · 풀링)과 정확히 짝을 이룬다. 셋째는 선형 모형을 배제한다.

In [ ]:
PATCH = 3
def make_templates():
    h = np.zeros((3,3)); h[1,:] = 1.0
    v = np.zeros((3,3)); v[:,1] = 1.0
    return [ (t - t.mean())/np.linalg.norm(t - t.mean()) for t in (h, v) ]
T0, T1 = make_templates()
assert abs(float(np.sum(T0*T1))) < 1e-12, "두 무늬는 직교해야 한다"

def sample_imgs(n, rng):
    y = rng.integers(0, 2, size=n)
    x = rng.normal(0, 1.0, size=(n, IMG, IMG))
    lo = IMG - PATCH + 1
    r = rng.integers(0, lo, n); c = rng.integers(0, lo, n)
    s = rng.choice([-1.0, 1.0], n)
    for i in range(n):
        x[i, r[i]:r[i]+PATCH, c[i]:c[i]+PATCH] += AMP * s[i] * (T1 if y[i] == 1 else T0)
    return x, y

xs, ys = sample_imgs(8, np.random.default_rng(SEED))
fig, axes = plt.subplots(2, 4, figsize=(7.6, 4.0))
for ax, im, yy in zip(axes.ravel(), xs, ys):
    ax.imshow(im, cmap='gray'); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title(lab(f'{"세로" if yy else "가로"}', f'{"vert" if yy else "horiz"}'), fontsize=9)
fig.suptitle(lab('같은 무늬가 매번 다른 위치에, 밝게 또는 어둡게 찍힌다',
                 'the same motif appears at a random position, bright or dark'), y=1.02, fontsize=10)
show('task_examples')

---
## 4. 세 가설 공간 — 나머지를 완전히 고정하기

§1.4.2 4절이 경고했듯 구조와 최적화는 얽히기 쉽다. 그래서 **최적화를 아예 제거한다.**
세 경우 모두 특징 사상 $\phi$를 고정하고 **닫힌 형태의 릿지 회귀**를 푼다. 알고리즘이 문자 그대로 동일하므로
차이가 나면 그것은 오직 $\phi$ 때문이다.

| 이름 | $\phi(x)$ | 특징 수 |
|---|---|---|
| 선형 | 화소 그대로 | 100 |
| MLP-무작위 | $\mathrm{ReLU}(Wx + b)$, $W$ 무작위 밀집 | **512** |
| CNN-무작위 | 무작위 $3\times3$ 필터 96개 → ReLU → 공간 최대·평균 풀링 | 192 |

**MLP에 특징을 더 많이 줬다는 점이 중요하다.** 그래도 CNN이 이기면 그것은 용량 때문이 아니라 **편향이 맞아서**다.

무작위 특징을 쓰는 것은 §35(게으른 학습)에서 다룰 체제에 해당한다. 학습된 특징을 쓰는 실제 신경망과 다르며,
그 차이는 이 노트북의 범위 밖이다.

In [ ]:
def im2col(x, k=PATCH):
    n, H, _ = x.shape; o = H - k + 1
    out = np.empty((n, o, o, k*k))
    for i in range(k):
        for j in range(k):
            out[:, :, :, i*k + j] = x[:, i:i+o, j:j+o]
    return out

_r = np.random.default_rng(SEED)
D  = IMG*IMG
W_MLP = _r.normal(0, 1/np.sqrt(D), (D, 512)); B_MLP = _r.normal(0, 0.3, 512)
F_CNN = _r.normal(0, 1, (PATCH*PATCH, 96)); F_CNN /= np.linalg.norm(F_CNN, axis=0, keepdims=True)
B_CNN = _r.normal(0, 0.3, 96)

def phi_linear(x):
    return x.reshape(len(x), -1)

def phi_mlp(x):
    return np.maximum(x.reshape(len(x), -1) @ W_MLP + B_MLP, 0)

def phi_cnn(x):
    r = np.maximum(im2col(x) @ F_CNN + B_CNN, 0).reshape(len(x), -1, 96)
    return np.concatenate([r.max(axis=1), r.mean(axis=1)], axis=1)

PHI = {'선형': phi_linear, 'MLP': phi_mlp, 'CNN': phi_cnn}
LAMS = 10.0 ** np.arange(-4, 6)

def ridge_acc(Ftr, ytr, Fva, yva, Fte, yte, lams=LAMS):
    # 고유분해를 한 번만 해서 lambda 전체를 싸게 훑는다.
    ad = lambda F: np.hstack([F, np.ones((len(F), 1))])
    Ftr, Fva, Fte = ad(Ftr), ad(Fva), ad(Fte)
    ev, V = np.linalg.eigh(Ftr.T @ Ftr)
    Vb = V.T @ (Ftr.T @ (2.0*ytr - 1.0))
    best, bw = -1.0, None
    for lam in lams:                       # lambda 선택은 검증 집합에서 (§1.1.6)
        w = V @ (Vb/(ev + lam))
        a = float(np.mean((Fva @ w > 0) == (yva == 1)))
        if a > best:
            best, bw = a, w
    return float(np.mean((Fte @ bw > 0) == (yte == 1)))   # 보고는 시험 집합에서

n_ev = 1500 if FAST else 3000
x_te, y_te = sample_imgs(n_ev, np.random.default_rng(999))
x_va, y_va = sample_imgs(n_ev//2, np.random.default_rng(998))
print(f"검증 {len(y_va):,}개 · 시험 {len(y_te):,}개 준비 완료")
print("릿지 정칙화 계수는 검증 집합에서 고르고 정확도는 시험 집합에서 보고한다 (§1.1.6)")

### 4.1 선형 모형은 왜 우연 수준에 머무는가

실험 전에 예측할 수 있다. 부호 $s$가 $\pm 1$ 균등이므로, 각 부류의 조건부 분포가 $x \mapsto -x$ 에 대해 **대칭**이다.

> **명제.** 편향이 없는 선형 규칙 $\mathrm{sign}(w^{\top}x)$ 의 정확도는 모든 $w$에 대해 정확히 $1/2$ 이다.
>
> **증명.** 각 $y$에 대해 $w^{\top}x \mid y$ 의 분포가 0에 대해 대칭이므로 $\mathbb{P}(w^{\top}x > 0 \mid y) = 1/2$ 이다.
> 따라서 정확도는 $\tfrac12 \cdot \tfrac12 + \tfrac12 \cdot \tfrac12 = \tfrac12$. $\blacksquare$

편향 항을 허용하면 두 부류의 **퍼짐**이 다른 것을 이용할 여지가 원리적으로는 있다. 다만 그것은
$\lvert w^{\top}x \rvert$ 의 크기를 보는 판정이라 단조인 선형 규칙으로는 구현되지 않는다. 아래에서 실제로 우연 수준에 머문다.

---
## 5. 표본 효율

$n$을 훑으며 시험 정확도를 잰다. **자료·손실·최적화기·검증 절차가 전부 동일**하고 $\phi$만 다르다.

In [ ]:
NS = [3, 5, 10, 20, 40, 80, 160, 320, 640, 1280] + ([] if FAST else [2560, 5120])
NTR = 2 if FAST else 3

def sweep(feat_of, tag=''):
    out = {k: np.zeros(len(NS)) for k in PHI}
    Fva = {k: feat_of[k](x_va) for k in PHI}
    Fte = {k: feat_of[k](x_te) for k in PHI}
    for i, n in enumerate(NS):
        for k in PHI:
            acc = []
            for t in range(NTR):
                xt, yt = sample_imgs(n, np.random.default_rng(100 + t))
                acc.append(ridge_acc(feat_of[k](xt), yt, Fva[k], y_va, Fte[k], y_te))
            out[k][i] = np.mean(acc)
    return out

res = sweep(PHI)
print("     n     선형     MLP     CNN")
for i, n in enumerate(NS):
    print(f"{n:>6}  " + "  ".join(f"{res[k][i]:.3f}" for k in ['선형','MLP','CNN']))

TARGET = 0.70
print(f"\n정확도 {TARGET:.0%} 도달에 필요한 n")
need = {}
for k in ['선형','MLP','CNN']:
    hit = [n for n, a in zip(NS, res[k]) if a >= TARGET]
    need[k] = hit[0] if hit else None
    print(f"  {k:>4}: " + (f"{hit[0]}개 이하" if hit else f"{NS[-1]}개로도 도달 못 함"))
if need['CNN'] and need['MLP']:
    print(f"\n  -> MLP는 CNN보다 최소 {need['MLP']//need['CNN']}배 많은 표본이 필요하다.")

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 4.0))
for k, c, m in [('선형', CB[5], 'o'), ('MLP', CB[3], 's'), ('CNN', CB[4], '^')]:
    ax.semilogx(NS, res[k], m + '-', ms=5, color=c,
                label=lab({'선형':'선형 (특징 100)','MLP':'MLP-무작위 (512)','CNN':'CNN-무작위 (192)'}[k],
                          {'선형':'linear (100)','MLP':'MLP-random (512)','CNN':'CNN-random (192)'}[k]))
ax.axhline(0.5, color=CB[0], lw=0.8, ls=':')
ax.axhline(TARGET, color=CB[0], lw=1.0, ls='--',
           label=lab(f'목표 {TARGET:.0%}', f'target {TARGET:.0%}'))
ax.set_ylim(0.45, 1.02)
ax.set_xlabel(lab('훈련 표본 수 $n$ (개)', 'training samples $n$'))
ax.set_ylabel(lab('시험 정확도', 'test accuracy'))
ax.set_title(lab('자료·손실·최적화기 고정, 가설 공간만 변경',
                 'same data, loss, and optimizer - only the hypothesis space differs'), fontsize=10)
ax.legend(fontsize=8, loc='center right')
show('sample_efficiency')

> ### 읽는 법
>
> **CNN은 표본 몇 개로 끝난다.** MLP는 특징을 두 배 넘게 갖고도 훨씬 느리게 오른다.
> 선형은 4.1절의 명제대로 끝까지 $0.5$ 근처에 머문다.
>
> 세 곡선의 간격은 **자료로 메울 수 없다.** §1.2.4에서 손실을 바꿨을 때 점선이 평평했던 것과 같은 종류의 격차이며,
> 다만 그때는 ②(목적함수)가 만든 것이고 여기서는 ①(구조)이 만든 것이다.

---
## 6. CNN이 좋은 것인가, 편향이 맞는 것인가

모든 이미지에 **같은** 화소 순열 $P$ 를 적용한다. 정보량은 전혀 줄지 않는다 — 순열은 가역이다.
바뀌는 것은 "이웃"이라는 개념뿐이다.

두 가지를 미리 증명할 수 있다.

> **명제 1.** 선형 특징에 대한 릿지 회귀의 예측은 화소 순열에 **정확히 불변**이다.
>
> **증명.** $P$ 는 직교행렬이다. 순열된 자료 행렬은 $X' = XP^{\top}$ 이고
> $w' = (PX^{\top}XP^{\top} + \lambda I)^{-1}PX^{\top}t = P(X^{\top}X + \lambda I)^{-1}X^{\top}t = Pw$ 이다.
> 따라서 $x'^{\top}w' = (Px)^{\top}(Pw) = x^{\top}w$. $\blacksquare$
>
> **명제 2.** MLP-무작위 특징의 성능은 순열에 대해 **분포적으로 불변**이다. $W$ 가 등방 무작위 행렬이므로
> $PW$ 와 $W$ 의 분포가 같기 때문이다. (한 번 뽑힌 $W$ 를 고정하면 정확히 같지는 않고 표집 오차만큼 흔들린다.)

**CNN에 대해서는 아무 보장이 없다.** 국소성과 가중치 공유는 화소의 배열에 의존하기 때문이다.

In [ ]:
PERM = np.random.default_rng(SEED + 7).permutation(D)
def permute(x):
    return x.reshape(len(x), -1)[:, PERM].reshape(-1, IMG, IMG)

PHI_P = {k: (lambda f: (lambda x: f(permute(x))))(PHI[k]) for k in PHI}
res_p = sweep(PHI_P)

print("           원본                   순열 적용")
print("     n    선형   MLP   CNN      선형   MLP   CNN")
for i, n in enumerate(NS):
    a = "  ".join(f"{res[k][i]:.3f}" for k in ['선형','MLP','CNN'])
    b = "  ".join(f"{res_p[k][i]:.3f}" for k in ['선형','MLP','CNN'])
    print(f"{n:>6}   {a}    {b}")

# 명제 1의 수치 확인
assert np.allclose(res['선형'], res_p['선형']), "선형은 순열에 정확히 불변이어야 한다"
print("\n명제 1 확인: 선형 결과가 두 조건에서 완전히 동일하다.")
print(f"명제 2 확인: MLP 최종 정확도 {res['MLP'][-1]:.3f} vs {res_p['MLP'][-1]:.3f} (표집 오차 범위)")
print(f"대비:        CNN 최종 정확도 {res['CNN'][-1]:.3f} vs {res_p['CNN'][-1]:.3f}  <- 무너진다")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.8), sharey=True)
for ax, r, ttl in zip(axes, [res, res_p],
                      [lab('원본 이미지', 'original images'),
                       lab('화소를 고정 순열로 섞음 (정보량 동일)', 'fixed pixel permutation (same information)')]):
    for k, c, m in [('선형', CB[5], 'o'), ('MLP', CB[3], 's'), ('CNN', CB[4], '^')]:
        ax.semilogx(NS, r[k], m + '-', ms=5, color=c, label=k if KO_FONT else
                    {'선형':'linear','MLP':'MLP','CNN':'CNN'}[k])
    ax.axhline(0.5, color=CB[0], lw=0.8, ls=':')
    ax.set_xlabel(lab('훈련 표본 수 $n$ (개)', 'training samples $n$'))
    ax.set_title(ttl, fontsize=10); ax.set_ylim(0.45, 1.02)
axes[0].set_ylabel(lab('시험 정확도', 'test accuracy')); axes[1].legend(fontsize=8)
fig.suptitle(lab('합성곱의 이점은 화소 배열에 의존한다 — 구조가 좋은 것이 아니라 문제에 맞았던 것',
                 'the convolutional advantage depends on pixel layout'), y=1.03, fontsize=10)
show('permutation_control')

> ### 이 그림이 §1.4.4의 결론이다
>
> 순열은 가역이므로 §1.2.6의 정리에 따라 **베이즈 위험 $R^{*}$ 는 조금도 변하지 않는다.**
> 그런데 CNN의 표본 효율은 무너지고 MLP는 그대로다.
>
> 따라서 CNN의 이점은 **모형의 우수성이 아니라 편향과 문제의 일치**에서 온다.
> "좋은 구조"라는 것은 없고 **"이 문제에 맞는 구조"**가 있을 뿐이다 — §1.5가 이것을 정리로 만든다.

---
## 7. 자기 점검

1. 1절에서 기저를 바꾸자 $f(2)$ 가 10.5에서 24로 바뀌었다. 이것은 §1.4.2의 세 출처 중 **어느 것**의 변화인가? 하나로 답할 수 있는가?
2. 5절에서 `AMP` 를 3으로 낮추면 세 곡선은 어떻게 될까? 간격이 **좁아지겠는가 넓어지겠는가**? 예측한 뒤 바꿔 확인하라.
3. 6절에서 CNN이 순열 후에도 우연보다는 나은 이유는 무엇인가? (힌트: 순열이 모든 이웃 관계를 파괴하는가?)
4. MLP-무작위의 특징 수를 512에서 4096으로 늘리면 CNN을 따라잡겠는가? 용량과 편향 중 무엇이 병목인가?

In [ ]:
# 자기 점검 4의 확인 — 용량을 8배로 늘려 본다
_r2 = np.random.default_rng(SEED + 99)
W_BIG = _r2.normal(0, 1/np.sqrt(D), (D, 4096)); B_BIG = _r2.normal(0, 0.3, 4096)
phi_big = lambda x: np.maximum(x.reshape(len(x), -1) @ W_BIG + B_BIG, 0)

n_chk = 640
Fva_b, Fte_b = phi_big(x_va), phi_big(x_te)
acc = []
for t in range(2):
    xt, yt = sample_imgs(n_chk, np.random.default_rng(100 + t))
    acc.append(ridge_acc(phi_big(xt), yt, Fva_b, y_va, Fte_b, y_te))
i = NS.index(n_chk)
print(f"n={n_chk}에서")
print(f"  MLP  512 특징: {res['MLP'][i]:.3f}")
print(f"  MLP 4096 특징: {np.mean(acc):.3f}")
print(f"  CNN  192 특징: {res['CNN'][i]:.3f}")
print("\n-> 용량을 8배로 늘려도 특징이 1/20인 CNN에 미치지 못한다. 병목은 용량이 아니라 편향이다.")

---
## 8. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `AMP` | 0절 | 6.0 | 무늬 세기. 낮추면 과제가 어려워지고 CNN 곡선도 내려온다 |
| `IMG` | 0절 | 10 | 이미지 크기. 키우면 위치 수가 늘어 MLP가 더 불리해진다 |
| `NS` | 5절 | 3~5120 | 표본 수 격자 |
| 무늬 정의 | 3절 | 가로/세로 막대 | 국소적이지 **않은** 무늬로 바꾸면 CNN의 이점이 줄어든다 |
| `W_MLP` 폭 | 4절 | 512 | 용량 대 편향 실험 |
| `SAVE_PDF` | 0절 | False | 그림을 벡터 PDF로 저장 |

**권하는 첫 실험** — 3절의 무늬를 **전역적인 것**으로 바꾸십시오. 예컨대 이미지 전체의 좌우 절반 평균 차이를
레이블로 삼으면, 국소성 가정이 무의미해져 CNN의 이점이 사라지고 선형 모형이 최고가 됩니다.
**구조에 절대적 우열이 없다는 것**을 확인하는 가장 빠른 방법입니다.

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")